In [3]:
#The purpose of this notebook is to optimize and XGBoost regressor on each of the 4 different feature sets, Peng, Ghorbani, Xiong and CHALPHAD.
#major change, Since all features are possible to calculate except for Ghorbani, Ghorbani alloys will be used and features calcuated for all 4 feature sets, and then the best model will be selected based on the performance on the Ghorbani alloys.


In [4]:
#import necessary libraries
#import the required libraries
import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
import re
from sklearn.model_selection import RepeatedKFold,ShuffleSplit
from CBFV import composition
from scipy.stats import sem
import warnings
import ujson as js
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

In [5]:
#Take formula column and parse it into a dataframe of element columns with atomic percentages as values. This function should be able to handle the complex formulas in the Ghorbani dataset, including nested parentheses, brackets, and braces, as well as fractions and equal splits.
def assemble_composition_df(df, formula_column):
    element_list = [
        "Ag", "Al", "Am", "As", "Au",
        "B", "Ba", "Be", "Bi",
        "C", "Ca", "Cd", "Ce", "Co", "Cr", "Cs", "Cu",
        "Dy",
        "Er", "Eu",
        "Fe",
        "Ga", "Gd", "Ge",
        "H", "Hf", "Hg", "Ho",
        "In", "Ir",
        "K",
        "La", "Li", "Lu",
        "Mg", "Mn", "Mo",
        "N", "Na", "Nb", "Nd", "Ni", "Np",
        "O", "Os",
        "P", "Pa", "Pb", "Pd", "Pr", "Pt", "Pu",
        "Rb", "Re", "Rh", "Ru",
        "S", "Sb", "Sc", "Se", "Si", "Sm", "Sn", "Sr",
        "Ta", "Tb", "Tc", "Te", "Th", "Ti", "Tl", "Tm",
        "U",
        "V",
        "W",
        "Y", "Yb",
        "Zn", "Zr"
    ]
    
    def parse_fraction(s):
        """Parse a string that might be a fraction (e.g., '5/6') or a number."""
        if '/' in s:
            num, denom = s.split('/')
            return float(num) / float(denom)
        return float(s)
    
    def parse_element_composition(formula_str):
        """
        Parse element-number pairs from a formula string.
        Returns a dict of {element: amount}
        """
        composition = {}
        # Pattern to match element followed by optional number (including fractions)
        pattern = r'([A-Z][a-z]?)(\d+(?:\.\d+)?(?:/\d+(?:\.\d+)?)?)?'
        
        matches = re.findall(pattern, formula_str)
        for element, amount in matches:
            if element and element in element_list:
                if amount:
                    val = parse_fraction(amount)
                else:
                    val = 1.0
                composition[element] = composition.get(element, 0) + val
        
        return composition
    
    def parse_formula(formula):
        """
        Parse a complete alloy formula handling nested brackets, parentheses, and braces.
        Returns a dict of {element: atomic_percent}
        """
        composition = {}
        
        # Remove citation references like [24], [30], etc. at the end
        formula = re.sub(r'\[\d+\]$', '', formula)
        formula = re.sub(r'\[\d+\]', '', formula)
        
        # Remove spaces and commas used as separators
        formula = formula.replace(' ', '').replace(',', '')
        
        def process_innermost_group(f):
            """Find and process the innermost bracketed group."""
            pattern = r'([\(\[\{])([^\(\)\[\]\{\}]+)([\)\]\}])(\d+(?:\.\d+)?)?'
            
            match = re.search(pattern, f)
            if not match:
                return f, False
            
            open_bracket, content, close_bracket, multiplier = match.groups()
            
            # Parse the content of the group
            inner_comp = parse_element_composition(content)
            
            # Calculate the sum of inner compositions
            inner_sum = sum(inner_comp.values())
            
            # Determine the multiplier
            if multiplier:
                mult = float(multiplier)
            else:
                mult = 1.0
            
            # Determine if inner values are fractions or percentages
            # If sum is close to 1, treat as fractions; if close to 100, treat as percentages
            if inner_sum > 1.5:  # Likely percentages within the group
                # Normalize to fractions, then multiply
                inner_comp = {k: v / inner_sum for k, v in inner_comp.items()}
            
            # Apply multiplier
            expanded = {elem: amt * mult for elem, amt in inner_comp.items()}
            
            # Create replacement string
            replacement_parts = []
            for elem, amt in expanded.items():
                replacement_parts.append(f"{elem}{amt}")
            replacement = ''.join(replacement_parts)
            
            new_f = f[:match.start()] + replacement + f[match.end():]
            
            return new_f, True
        
        # Iteratively process innermost groups until none remain
        processed_formula = formula
        max_iterations = 20
        iteration = 0
        
        while iteration < max_iterations:
            processed_formula, found = process_innermost_group(processed_formula)
            if not found:
                break
            iteration += 1
        
        # Now parse the final expanded formula
        composition = parse_element_composition(processed_formula)
        
        # Handle equal split case (elements with no numbers)
        total = sum(composition.values())
        num_elements = len(composition)
        
        # Check if all elements have value 1.0 (no numbers given)
        if num_elements > 0 and all(v == 1.0 for v in composition.values()):
            equal_share = 100.0 / num_elements
            composition = {k: equal_share for k in composition}
        # If total is very small (< 2), scale up to 100
        elif total > 0 and total < 2:
            scale = 100.0 / total
            composition = {k: v * scale for k, v in composition.items()}
        
        return composition
    
    # Process all formulas
    composition_dicts = []
    for formula in df[formula_column]:
        try:
            comp = parse_formula(str(formula))
            composition_dicts.append(comp)
        except Exception as e:
            print(f"Error parsing '{formula}': {e}")
            composition_dicts.append({})
    
    # Create DataFrame with element columns
    comp_df = pd.DataFrame(composition_dicts)
    
    # Ensure all element columns exist, fill missing with 0
    for elem in element_list:
        if elem not in comp_df.columns:
            comp_df[elem] = 0.0
    
    # Reorder columns to match element_list and fill NaN with 0
    comp_df = comp_df.reindex(columns=element_list, fill_value=0.0)
    comp_df = comp_df.fillna(0.0)
    
    return comp_df

#take composition df and produce composition strings
def canonical_comp_string(df, tol=1e-9, decimals=2):
    element_cols = sorted([c for c in df.columns if c != "Composition String"])
    out = []
    for _, row in df[element_cols].iterrows():
        vals = row.astype(float).fillna(0.0).to_numpy()
        vals[vals < tol] = 0.0
        s = vals.sum()
        if s <= 0:
            out.append("")
            continue
        vals = vals / s * 100.0
        vals = np.round(vals, decimals)
        parts = [f"{el}{v:.{decimals}f}" for el, v in zip(element_cols, vals) if v > 0]
        out.append("".join(parts))
    return out

#create function that takes a composition df and gets the index of rows who sum to greater than 100. This is to catch any errors in the composition parsing where the percentages add up to more than 100.
def find_rows_sum_greater_than_100(df):
    over_100_index = df.sum(axis=1) > 100
    return over_100_index



In [6]:
#load and process the Ghorbani dataset 
raw_Ghorbani_df = pd.read_excel(r"Data\Paper Data\Ghorbani, 2022.xlsx")

#Convert the alloys in Ghorbani dataset into cananical composition strings
Ghorbani_composition_df = assemble_composition_df(raw_Ghorbani_df, "Alloy")
Ghorbani_canonical_strings = canonical_comp_string(Ghorbani_composition_df)

#save the original length of the Ghorbani dataset for later comparison after dropping rows that sum to greater than 100
original_Ghorbani_length = len(raw_Ghorbani_df)
print(f"Original length of Ghorbani dataset: {original_Ghorbani_length}")

#replace the Ghorbani alloy column with the canonical composition strings and rename as composition string
raw_Ghorbani_df["Composition String"] = Ghorbani_canonical_strings
raw_Ghorbani_df = raw_Ghorbani_df.drop(columns=["Alloy"])


#get the index of any rows in the composition df that sum to greater than 100
Ghorbani_over_100_index = find_rows_sum_greater_than_100(Ghorbani_composition_df)
print(f"Number of rows in Ghorbani composition df that sum to greater than 100: {Ghorbani_over_100_index.sum()}")

#drop the rows over 100 from the composition df and the original df
raw_Ghorbani_df = raw_Ghorbani_df[~Ghorbani_over_100_index].reset_index(drop=True)
Ghorbani_composition_df = Ghorbani_composition_df[~Ghorbani_over_100_index].reset_index(drop=True)

#Check for any duplicate composition strings in the Ghorbani dataset and replace with the mean of the duplicates
raw_Ghorbani_df = raw_Ghorbani_df.groupby("Composition String").mean().reset_index()
Ghorbani_composition_df['Composition String'] = raw_Ghorbani_df['Composition String']
Ghorbani_composition_df = Ghorbani_composition_df.groupby("Composition String").mean().reset_index()

#calculate the post processing length of the Ghorbani dataset and print the number of rows dropped
post_processing_Ghorbani_length = len(raw_Ghorbani_df)

#cacluate the number of duplicate rows in the Ghorbani dataset and print
ghorbani_duplicate_count = original_Ghorbani_length - post_processing_Ghorbani_length - Ghorbani_over_100_index.sum()
print(f"Number of duplicate rows in Ghorbani dataset that were averaged: {ghorbani_duplicate_count}")
print(f"Number of rows dropped from Ghorbani dataset after processing: {original_Ghorbani_length - post_processing_Ghorbani_length}")

#create a set of the Ghorbani composition strings for later comparison with the other datasets
Ghorbani_canonical_strings = raw_Ghorbani_df["Composition String"].tolist()

#split the Ghorbani dataset into X data
Ghorbani_X = raw_Ghorbani_df.drop(columns=["No.", "Tg", "Tx", "Tl"])

#print the final length of the Ghorbani dataset after processing
print(f"Final length of Ghorbani dataset after processing: {len(raw_Ghorbani_df)}")

Original length of Ghorbani dataset: 715
Number of rows in Ghorbani composition df that sum to greater than 100: 26
Number of duplicate rows in Ghorbani dataset that were averaged: 28
Number of rows dropped from Ghorbani dataset after processing: 54
Final length of Ghorbani dataset after processing: 661


In [7]:
#find the interseciton of Xiong and Ghorbani composition strings, only use these for the rest of the analysis to ensure a fair comparison between the two datasets. This is because the Ghorbani dataset is the smallest and we want to make sure we are comparing the same alloys across all datasets.
raw_xiong_df = pd.read_excel(r"Data\Paper Data\XIONG 2021.xlsx")

xiong_comp_df = assemble_composition_df(raw_xiong_df, "Alloys")

xiong_can_strings = canonical_comp_string(xiong_comp_df)

print(len(xiong_can_strings))
print(len(Ghorbani_canonical_strings))

# Find strings in Ghorbani_canonical_strings that are not in xiong_can_strings
ghorbani_not_in_xiong = set(Ghorbani_canonical_strings) - set(xiong_can_strings)
print(f"\n{len(ghorbani_not_in_xiong)} strings in Ghorbani_canonical_strings not in xiong_can_strings:")

    
#find the intersection of the two sets
intersection = set(Ghorbani_canonical_strings).intersection(set(xiong_can_strings))
print(f"\nNumber of composition strings in the intersection of Ghorbani and Xiong datasets: {len(intersection)}")

#filter the Ghorbani, X canonical strings, and composition df to only include the intersection
Ghorbani_X = Ghorbani_X[Ghorbani_X["Composition String"].isin(intersection)].reset_index(drop=True)
Ghorbani_canonical_strings = Ghorbani_X["Composition String"].tolist()
Ghorbani_composition_df = Ghorbani_composition_df[Ghorbani_composition_df["Composition String"].isin(intersection)].reset_index(drop=True)


695
661

204 strings in Ghorbani_canonical_strings not in xiong_can_strings:

Number of composition strings in the intersection of Ghorbani and Xiong datasets: 457


In [8]:
#Peng uses a basic compositional input, convert the ghorbani composition strings into a composition df and use this to make Peng dataframe 
#peng composition df is made by taking a copy of the Ghorbani composition df
Peng_composition_df = Ghorbani_composition_df.copy()
Peng_canonical_strings = Ghorbani_canonical_strings.copy()

#print the length of the Peng composition df and the length of peng_canonical_strings 
print(f"Length of Peng composition df: {len(Peng_composition_df)}")
print(f"Length of Peng canonical strings: {len(Peng_canonical_strings)}")

#processing already occured so just checking that nothing is amiss with the Peng composition df and canonical strings before using them to create the Peng dataset
#Check for any rows in the Peng composition df that sum to greater than 100 and print
Peng_over_100_index = find_rows_sum_greater_than_100(Peng_composition_df.drop(columns=["Composition String"]))
print(f"Number of rows in Peng composition df that sum to greater than 100: {Peng_over_100_index.sum()}")

#find duplicates in the Peng composition df and print the number of duplicates
Peng_duplicates = Peng_composition_df.duplicated(subset=["Composition String"], keep=False)
print(f"Number of duplicate rows in Peng composition df: {Peng_duplicates.sum()}")

#normalize the Peng composition df so that all rows sum to 1
Peng_composition_df = Peng_composition_df.drop(columns=["Composition String"])
Peng_composition_df = Peng_composition_df.div(100, axis=0)

#drop columns with all zeros from the Peng composition df
pre_column_drop_length = len(Peng_composition_df.columns)
Peng_composition_df = Peng_composition_df.loc[:, (Peng_composition_df != 0).any(axis=0)]
post_column_drop_length = len(Peng_composition_df.columns)
print(f"Dropped {pre_column_drop_length - post_column_drop_length} columns with all zeros from Peng composition df")

#print the length of the Peng composition df and the number of unique composition strings in the Peng canonical strings to check for duplicates
print(f"Length of Peng composition df: {len(Peng_composition_df)}")

#create the peng x data
Peng_x = Peng_composition_df.copy()

Length of Peng composition df: 457
Length of Peng canonical strings: 457
Number of rows in Peng composition df that sum to greater than 100: 0
Number of duplicate rows in Peng composition df: 0
Dropped 40 columns with all zeros from Peng composition df
Length of Peng composition df: 457


In [9]:
#define function to compute xiong features from a composition dataframe and an elemental property dataframe. This function should be able to handle missing values in the elemental property dataframe by either dropping those elements or filling with NaN, depending on the missing_policy parameter. It should also have an option to compute features using only nonzero constituents or all constituents, controlled by the use_nonzero_only parameter. The function should return a new dataframe with the computed features for each alloy.
def compute_xiong_features(
    alloy_df,
    elem_info_df,
    element_col="Element",
    radius_col="Rm (nm)",
    missing_policy="warn",
    use_nonzero_only=True,
    verbose=False,
):
    """
    Compute Xiong-style alloy descriptors from a composition DataFrame and
    an elemental-property DataFrame.

    Assumes alloy_df already contains normalized atomic fractions:
        - each row is one alloy
        - columns are element symbols
        - row sums should be ~1
    """

    # 1. Clean and align
    ei = elem_info_df.copy()
    ei.columns = ei.columns.astype(str).str.strip()
    ei[element_col] = ei[element_col].astype(str).str.strip()
    ei = ei.set_index(element_col)

    alloy_df = alloy_df.copy()
    alloy_df.columns = alloy_df.columns.astype(str).str.strip()

    alloy_elements = set(alloy_df.columns)
    known_elements = set(ei.index)
    missing = sorted(alloy_elements - known_elements)

    if missing:
        if missing_policy == "error":
            raise ValueError(
                f"Elements in alloy_df not found in elem_info_df: {missing}"
            )
        elif missing_policy == "warn":
            warnings.warn(
                f"Elements in alloy_df not found in elem_info_df (dropped): {missing}"
            )
        elif missing_policy != "drop":
            raise ValueError("missing_policy must be 'warn', 'drop', or 'error'")

    common_elements = sorted(alloy_elements & known_elements)
    if not common_elements:
        raise ValueError("No overlapping elements between alloy_df and elem_info_df.")

    A = alloy_df[common_elements].copy().fillna(0.0)
    A = A.apply(pd.to_numeric, errors="coerce").fillna(0.0)

    row_sums = A.sum(axis=1)
    zero_rows = row_sums <= 0
    if zero_rows.any():
        raise ValueError(
            f"{int(zero_rows.sum())} alloy row(s) sum to 0; cannot compute features."
        )

    P = ei.loc[common_elements].copy()
    P = P.apply(pd.to_numeric, errors="coerce")

    A_mat = A.to_numpy(dtype=np.float64)
    P_mat = P.to_numpy(dtype=np.float64)
    prop_names = P.columns.tolist()

    n_alloys, n_elems = A_mat.shape
    n_props = P_mat.shape[1]

    if verbose:
        print(f"Elements matched: {len(common_elements)}  |  Dropped: {len(missing)}")

    constituent_mask = A_mat > 0 if use_nonzero_only else np.ones_like(A_mat, dtype=bool)

    x1 = np.full((n_alloys, n_props), np.nan, dtype=np.float64)
    x2 = np.full((n_alloys, n_props), np.nan, dtype=np.float64)
    xD = np.full((n_alloys, n_props), np.nan, dtype=np.float64)
    xd = np.full((n_alloys, n_props), np.nan, dtype=np.float64)

    # 2. Compute descriptors property-by-property
    for j in range(n_props):
        prop_vals = P_mat[:, j]
        prop_2d = np.broadcast_to(prop_vals, (n_alloys, n_elems))

        active = constituent_mask
        finite_prop = np.isfinite(prop_2d)

        # x1 = sum(a_i * x_i) / sum(a_i over valid constituents)
        x1_weights = np.where(active & finite_prop, A_mat, 0.0)
        x1_terms = np.where(active & finite_prop, A_mat * prop_2d, 0.0)
        x1_den = x1_weights.sum(axis=1)
        x1[:, j] = np.where(x1_den > 0, x1_terms.sum(axis=1) / x1_den, np.nan)

        # x2 = (sum(a_i / x_i))^-1 ; NaN if any constituent has x_i == 0 or NaN
        bad_x2 = active & ((~finite_prop) | (prop_2d == 0))
        any_bad_x2 = bad_x2.any(axis=1)

        good_x2 = active & finite_prop & (prop_2d != 0)
        with np.errstate(divide="ignore", invalid="ignore"):
            sum_a_over_x = np.where(good_x2, A_mat / prop_2d, 0.0).sum(axis=1)

        x2[:, j] = np.where(
            (~any_bad_x2) & np.isfinite(sum_a_over_x) & (sum_a_over_x != 0),
            1.0 / sum_a_over_x,
            np.nan,
        )

        # xD = sqrt(sum(a_i * (x_i - x1)^2))
        bad_xD = active & (~finite_prop)
        any_bad_xD = bad_xD.any(axis=1)

        diff_sq = (prop_2d - x1[:, [j]]) ** 2
        diff_sq = np.where(active & finite_prop, diff_sq, 0.0)
        xD_val = np.sqrt((A_mat * diff_sq).sum(axis=1))
        xD[:, j] = np.where(~any_bad_xD & np.isfinite(x1[:, j]), xD_val, np.nan)

        # xd = sqrt(sum(a_i * (1 - x_i/x1)^2))
        x1j = x1[:, j]
        x1_safe = np.where((x1j == 0) | (~np.isfinite(x1j)), np.nan, x1j)

        with np.errstate(divide="ignore", invalid="ignore"):
            ratio = prop_2d / x1_safe[:, None]

        rel_diff_sq = (1.0 - ratio) ** 2
        rel_diff_sq = np.where(np.isfinite(rel_diff_sq) & active & finite_prop, rel_diff_sq, 0.0)

        bad_xd = active & (~finite_prop)
        any_bad_xd = bad_xd.any(axis=1)

        xd_val = np.sqrt((A_mat * rel_diff_sq).sum(axis=1))
        xd[:, j] = np.where(
            (~any_bad_xd) & np.isfinite(x1j) & (x1j != 0),
            xd_val,
            np.nan,
        )

    # 3. Assemble results
    results = {}
    for j, prop in enumerate(prop_names):
        prop = str(prop).strip()
        results[f"{prop}_x1"] = x1[:, j]
        results[f"{prop}_x2"] = x2[:, j]
        results[f"{prop}_xD"] = xD[:, j]
        results[f"{prop}_xd"] = xd[:, j]

    # 4. Delta
    if radius_col is not None:
        radius_col = str(radius_col).strip()
        if radius_col in prop_names:
            r_idx = prop_names.index(radius_col)
            r_vals = P_mat[:, r_idx]
            r_2d = np.broadcast_to(r_vals, (n_alloys, n_elems))
            finite_r = np.isfinite(r_2d)
            active = constituent_mask

            r_weights = np.where(active & finite_r, A_mat, 0.0)
            r_terms = np.where(active & finite_r, A_mat * r_2d, 0.0)
            r_den = r_weights.sum(axis=1)
            r_bar = np.where(r_den > 0, r_terms.sum(axis=1) / r_den, np.nan)
            results["r_bar"] = r_bar

            bad_delta = active & (~finite_r)
            any_bad_delta = bad_delta.any(axis=1)

            r_bar_safe = np.where((r_bar == 0) | (~np.isfinite(r_bar)), np.nan, r_bar)
            with np.errstate(divide="ignore", invalid="ignore"):
                r_ratio = r_2d / r_bar_safe[:, None]

            delta_term = (1.0 - r_ratio) ** 2
            delta_term = np.where(np.isfinite(delta_term) & active & finite_r, delta_term, 0.0)

            delta = np.sqrt((A_mat * delta_term).sum(axis=1))
            delta = np.where(
                (~any_bad_delta) & np.isfinite(r_bar) & (r_bar != 0),
                delta,
                np.nan,
            )
            results["delta"] = delta
        else:
            warnings.warn(
                f"radius_col='{radius_col}' not found in elem_info_df. Skipping delta computation."
            )

    xiong_features_df = pd.DataFrame(results, index=alloy_df.index)

    if verbose:
        nan_counts = xiong_features_df.isna().sum()
        cols_with_nan = nan_counts[nan_counts > 0]
        if len(cols_with_nan):
            print(f"Features with NaNs ({len(cols_with_nan)}):")
            print(cols_with_nan.to_string())
        else:
            print("No NaN values in output.")

    return xiong_features_df

In [15]:
#load both Xiong datasets
Xiong_element_info_df = pd.read_excel(r"Data\Paper Data\Xiong element data.xlsx")
Xiong_raw_df = pd.read_excel(r"Data\Paper Data\XIONG 2021.xlsx")

#Create a Xiong composition df and canonical composition strings using the same functions as for the Ghorbani dataset. This is to ensure that the same parsing and processing is applied to both datasets for a fair comparison. The Xiong composition df will be used to compute the Xiong features and the canonical composition strings will be added to the original Xiong dataframe for later comparison with the other datasets.
Xiong_raw_composition_df = assemble_composition_df(Xiong_raw_df, "Alloys")
Xiong_raw_canonical_strings = canonical_comp_string(Xiong_raw_composition_df)
Xiong_raw_df["Composition String"] = Xiong_raw_canonical_strings
Xiong_raw_df = Xiong_raw_df.drop(columns=["Alloys"])

#filter the Xiong_raw_df to only include Ghorbani composition strings for a fair comparison between the two datasets. This is because the Ghorbani dataset is the smaller of the two and we want to make sure we are comparing the same alloys across all datasets.
Xiong_raw_df = Xiong_raw_df[Xiong_raw_df["Composition String"].isin(Ghorbani_canonical_strings)].reset_index(drop=True)
Xiong_raw_canonical_strings = Xiong_raw_df["Composition String"].tolist()

#print the length of the Xiong_raw_df and the number of unique composition strings in the Xiong_raw_canonical_strings to check for duplicates
print(f"Length of Xiong_raw_df: {len(Xiong_raw_df)}")
print(f"Number of unique composition strings in Xiong_raw_canonical_strings: {len(set(Xiong_raw_df['Composition String']))}")

#normalized the composition df so that all rows sum to 1
Xiong_composition_df = Xiong_raw_composition_df.copy()
Xiong_composition_df = Xiong_composition_df.div(100, axis=0)

#drop the columns with all 0s in the normalized composition df
xiong_pre_column_drop = len(Xiong_composition_df.columns)
Xiong_composition_df = Xiong_composition_df.loc[:, (Xiong_composition_df != 0).any(axis=0)]
xiong_post_column_drop = len(Xiong_composition_df.columns)
print(f"Dropped {xiong_pre_column_drop - xiong_post_column_drop} columns with all zeros from Xiong normalized composition df")


#find the index of rows in the Xiong composition df that sum to greater than 100 and drop from both composition df and original df
Xiong_over_100_index = find_rows_sum_greater_than_100(Xiong_composition_df)
print(f"Number of rows in Xiong composition df that sum to greater than 100: {Xiong_over_100_index.sum()}")


# ── Compute Xiong features ─────────────────────────────────────────────────
xiong_features_df = compute_xiong_features(
    alloy_df=Xiong_composition_df,
    elem_info_df=Xiong_element_info_df,
    element_col="Element",
    radius_col="Rm (nm)",       # already in fractions
    missing_policy="warn",
    use_nonzero_only=True,      # only constituent elements per alloy
    verbose=True,
)

print(f"\nShape: {xiong_features_df.shape}")

#drop the less useful features from the Xiong dataset to reduce the feature space and prevent overfitting likely what Xiong did in their original feature engineering
xiong_features_df = xiong_features_df.drop(columns=["sVEC_x2", "pVEC_x2", "pVEC_xd", "dVEC_x2"])


#add the entropy and enthalpy of mixing features to the Xiong features df
xiong_features_df['Hmix (kJ/mol)'] = Xiong_raw_df['Hmix (kJ/mol)']
xiong_features_df['Smix (J/K/mol)'] = Xiong_raw_df['Smix (J/K/mol)']

#add the canonical strings from the original df to the Xiong features df for later comparison with the other datasets
xiong_features_df['Composition String'] = Xiong_raw_df['Composition String']

#check the Xiong_features_df for duplicates using the composition string and drop any duplicates by averaging the features of the duplicates
xiong_features_df = xiong_features_df.groupby("Composition String").mean().reset_index()

#length of the Xiong dataset after processing
post_processing_Xiong_length = len(xiong_features_df)

#print the number of duplicate rows in the Xiong dataset that were averaged and the number of rows dropped from the Xiong dataset after processing
print(f"Number of rows in Xiong dataset after processing: {post_processing_Xiong_length}")

Xiong_X = xiong_features_df.copy()

Length of Xiong_raw_df: 460
Number of unique composition strings in Xiong_raw_canonical_strings: 457
Dropped 34 columns with all zeros from Xiong normalized composition df
Number of rows in Xiong composition df that sum to greater than 100: 0
Elements matched: 45  |  Dropped: 0
Features with NaNs (5):
sVEC_x2     30
pVEC_x2    695
pVEC_xd    232
dVEC_x2    635
dVEC_xd      1

Shape: (695, 94)
Number of rows in Xiong dataset after processing: 457


C:\Users\Chris\AppData\Local\Temp\ipykernel_23860\738415743.py:104: RuntimeWarning: divide by zero encountered in divide
  1.0 / sum_a_over_x,


In [ ]:
#Create canoncial string sets for each dataset to find the intersections and unique compositions between the datasets
Xiong_composition_strings_set = set(xiong_features_df["Composition String"])
Peng_composition_strings_set = set(raw_Peng_df["Composition String"])
Ghorbani_composition_strings_set = set(raw_Ghorbani_df["Composition String"])

#find the intersecting compositions
intersection_compositions = Xiong_composition_strings_set & Peng_composition_strings_set & Ghorbani_composition_strings_set
print(f"Number of intersecting compositions between all three datasets: {len(intersection_compositions)}")

#filter the datasets to only include the intersecting compositions for later comparison of model performance on the same compositions
Xiong_X_intersection = Xiong_X[Xiong_X["Composition String"].isin(intersection_compositions)].reset_index(drop=True)
Peng_X_intersection = Peng_x[raw_Peng_df["Composition String"].isin(intersection_compositions)].reset_index(drop=True)
Ghorbani_X_intersection = Ghorbani_X[raw_Ghorbani_df["Composition String"].isin(intersection_compositions)].reset_index(drop=True)

#use the raw xiong dataset to pull the dmax values using the intersecting composition strings
y_data = raw_Xiong_df[raw_Xiong_df["Composition String"].isin(intersection_compositions)][["Composition String", 'Dmax (mm)']].reset_index(drop=True)
#average the dmax values for any duplicate compositions in the y data
y_data = y_data.groupby("Composition String").mean().reset_index()

#check the length of all data to ensure they match
print(f"Length of Xiong intersection data: {len(Xiong_X_intersection)}")
print(f"Length of Peng intersection data: {len(Peng_X_intersection)}")
print(f"Length of Ghorbani intersection data: {len(Ghorbani_X_intersection)}")
print(f"Length of y data: {len(y_data)}")

Number of intersecting compositions between all three datasets: 410
Length of Xiong intersection data: 410
Length of Peng intersection data: 410
Length of Ghorbani intersection data: 410
Length of y data: 410


In [ ]:
#split into train_opt and test sets using a 80/20 split and a fixed random state for reproducibility
train_opt_compositions, test_compositions = train_test_split(
    list(intersection_compositions), test_size=0.2, random_state=42)

#separate the train_opt and test sets for each dataset using the composition strings
Xiong_X_train_opt = Xiong_X_intersection[Xiong_X_intersection["Composition String"].isin(train_opt_compositions)].reset_index(drop=True)
Xiong_X_test = Xiong_X_intersection[Xiong_X_intersection["Composition String"].isin(test_compositions)].reset_index(drop=True)

Peng_X_train_opt = Peng_X_intersection[Peng_X_intersection["Composition String"].isin(train_opt_compositions)].reset_index(drop=True)
Peng_X_test = Peng_X_intersection[Peng_X_intersection["Composition String"].isin(test_compositions)].reset_index(drop=True)

Ghorbani_X_train_opt = Ghorbani_X_intersection[Ghorbani_X_intersection["Composition String"].isin(train_opt_compositions)].reset_index(drop=True)
Ghorbani_X_test = Ghorbani_X_intersection[Ghorbani_X_intersection["Composition String"].isin(test_compositions)].reset_index(drop=True)

y_train_opt = y_data[y_data["Composition String"].isin(train_opt_compositions)].reset_index(drop=True)
y_test = y_data[y_data["Composition String"].isin(test_compositions)].reset_index(drop=True)

In [ ]:
#create a dataframe with the composition string and a false target varibale for CBFV featurization of the intersecting compositions
cbfv_intersection_df = pd.DataFrame({"formula": list(train_opt_compositions), "target": [0]*len(train_opt_compositions)})

#featurize the intersecting compositions using CBFV and drop the target variable
train_opt_cbfv, _, _, skipped = composition.generate_features(cbfv_intersection_df, elem_prop="magpie", drop_duplicates=False)

#check if any compositions were skipped in the featurization process and print them
if len(skipped) > 0:
    print(f"Compositions skipped in CBFV featurization: {skipped}") 
    
# Create 5 CV groups by clustering similar alloys together
# Standardize features for clustering
kmeans_scaler = StandardScaler()
kmeans_X_scaled = kmeans_scaler.fit_transform(train_opt_cbfv)

# Use KMeans to group similar alloys into 5 clusters
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
cv_groups = kmeans.fit_predict(kmeans_X_scaled)

# Add group assignments to a dataframe for reference
cv_group_df = pd.DataFrame({
    'formula': train_opt_compositions,
    'cv_group': cv_groups
})

print("CV Group distribution:")
print(cv_group_df['cv_group'].value_counts().sort_index())
print(f"\nTotal samples: {len(cv_groups)}")
cv_group_df.head(10)

Processing Input Data: 100%|██████████| 328/328 [00:00<00:00, 29815.82it/s]


	Featurizing Compositions...


Assigning Features...: 100%|██████████| 328/328 [00:00<00:00, 17258.78it/s]

	Creating Pandas Objects...
CV Group distribution:
cv_group
0     79
1    142
2     58
3     38
4     11
Name: count, dtype: int64

Total samples: 328


,formula,cv_group
0,Ca27.20Cu36.40Mg36.40,1
1,Ca36.40Cu45.50Mg18.10,1
2,Ag12.00Cu44.00Zr44.00,1
3,Cu37.04Ni7.06Sn2.00Ti44.10Zr9.80,1
4,B22.00Fe66.00W6.00Y6.00,0
5,Al10.00Ce68.00Cu20.00Si2.00,2
6,La20.00Mg50.00Ni30.00,3
7,B4.00C4.00Fe76.00Mo3.00P10.00Si3.00,0
8,B18.43Co27.65Cr4.00Fe41.47Nb3.84Si4.61,0
9,B17.00Fe74.00Nb6.00Y3.00,0
